# torch.compile + TF32 attempts for NO inference at N=1401 (Timon round-10, item 3 gap)

Timon: "there is probably still room for optimising the NO at inference. Could you please check this before considering
the 2.29s as the final inference number." -- the profiling cell diagnosed WHERE the time goes but never tried to make it
faster; a second-opinion review of that cell's own result correctly flagged this as the one remaining real gap.

**torch.compile real result (Omar's own A100 run, 2026-09-12)**: 2145.7ms vs. eager 2287.2ms -- **1.07x, correctness-verified** (output
relative difference 2.03e-6, floating-point noise level for fp32). A real but modest speedup, not the answer by itself.

**Added TF32 matmul precision** after that same run's own warning flagged it: "TensorFloat32 tensor cores available but not
enabled ... consider `torch.set_float32_matmul_precision('high')`." Targets the SAME dominant cost the profiler found (GEMM/
bmm/einsum) from a different angle -- Ampere's tensor cores execute fp32 matmuls several times faster at TF32 (19-bit
mantissa) precision. Tested both alone (eager+TF32) and combined with `torch.compile`.

**Correctness-checked before any speedup is trusted**: every variant's output is compared against the strict-fp32 eager
baseline. A failure in either `torch.compile` or the TF32 test is reported honestly (not hidden), and eager mode's number
stays the answer for whichever one fails.

* **NEEDS A GPU.**
* Takes ~15-20 minutes -- the first compiled call triggers real compilation, untimed (`compile_warmup`); TF32 adds two
  more full timing passes on top of the already-run eager/compile ones.
* Checkpoint path is guessed (`CKPT` near the top of the cell) -- update it if the assert fails.
* Saves a bar-chart figure (eager vs. compiled ms/sample, or just eager if compile failed).


In [ ]:
# =====================================================================
#  CELL -- torch.compile as an actual optimization attempt for NO
#  inference at N=1401, before treating 2.29s as the final number
#  (Timon round-10, item 3: "there is probably still room for
#  optimising the NO at inference. Could you please check this before
#  considering the 2.29s as the final inference number.")
#
#  The profiling cell (Round6_NO_Inference_Profile_N1401.ipynb) measured
#  WHERE the time goes (~90% in bmm/einsum/GEMM) but never tried to make
#  it faster. This cell tries the one safe, correctness-preserving
#  optimization that directly targets what the profiler's own numbers
#  pointed at: 4,800 cudaLaunchKernel calls / 2,893 "Command Buffer
#  Full" events across only 30 repeats -- torch.compile's kernel fusion
#  is built exactly for this kind of dispatch overhead, without
#  changing the architecture or a single weight.
#
#  Correctness-checked before any speedup is trusted: compares the
#  compiled model's own output against eager mode's on the same input.
#  If torch.compile fails or does not help on this architecture, that
#  is reported honestly -- eager mode stays the answer, not assumed
#  successful.
# =====================================================================
import json
import os
import subprocess
import sys


def run(cmd):
    print('$', ' '.join(str(c) for c in cmd), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    p.wait()
    if p.returncode != 0:
        raise subprocess.CalledProcessError(p.returncode, cmd)


from google.colab import drive
drive.mount('/content/drive')

REPO = '/content/OMAR'
if not os.path.isdir(REPO):
    run(['git', 'clone', '-b', 'claude/claude-code-question-d307wp',
         'https://github.com/SUHIBAMRO/OMAR.git', REPO])
else:
    run(['git', '-C', REPO, 'fetch', 'origin', 'claude/claude-code-question-d307wp'])
    run(['git', '-C', REPO, 'checkout', 'claude/claude-code-question-d307wp'])
    run(['git', '-C', REPO, 'reset', '--hard', 'origin/claude/claude-code-question-d307wp'])

WORK = f'{REPO}/Practical_Examples'
os.chdir(WORK)
sys.path.insert(0, WORK)
sys.path.insert(0, f'{WORK}/report_builders')

for _mod_name in list(sys.modules):
    if _mod_name == 'omar_pfem' or _mod_name.startswith('omar_pfem.'):
        del sys.modules[_mod_name]

import torch
assert torch.cuda.is_available(), 'this cell needs a real GPU'
print('GPU:', torch.cuda.get_device_name(0))
print('torch:', torch.__version__)

R = '/content/drive/MyDrive/pfem_run'
# BUG FOUND 2026-09-12: a hardcoded path here ('results/checkpoints/
# B1_neo_hookean/model_best.pt', which never existed on Drive) silently
# fell back to 'data_driven/B1_neo_hookean/model_best.pt' -- a COMPLETELY
# DIFFERENT model (train_data_driven.py's own data-driven-loss baseline
# from the round-5/6 comparison study). Same architecture as the
# data-driven model, so this cell's TIMING numbers are likely unaffected
# either way -- but re-run to be sure now that accuracy elsewhere was
# found to be corrupted by this same bug. Fixed properly this time:
# resolve by CONTENT (sha256), verified against the zero-shot study's own
# already-trusted checkpoint fingerprint, not by guessing a path -- see
# resolve_b1_checkpoint.py's own docstring.
from omar_pfem.resolve_b1_checkpoint import resolve_b1_neo_hookean_checkpoint
CKPT, _ckpt_fp = resolve_b1_neo_hookean_checkpoint(R)
print(f'Resolved checkpoint (verified by fingerprint): {CKPT}')
print('Using checkpoint:', CKPT)

device = torch.device('cuda')
dtype = torch.float32

from omar_pfem.resolution_invariance_zeroshot import build_sample_b1
from omar_pfem.measure_inference_latency import build_model
from omar_pfem.no_inference_torch_compile import profile_with_torch_compile
import argparse

args = argparse.Namespace(
    model='Transolver_Irregular_Mesh', n_hidden=256, n_layers=4, n_heads=8,
    mlp_ratio=2, dropout=0.1, unified_pos=0, ref=16, slice_num=128, fun_dim=4,
    use_soft_dirichlet=1, Lx=1.0, Ly=1.0, R_out=2.0,
)

model = build_model(args, device)
state_dict = torch.load(CKPT, map_location=device)
model.load_state_dict(state_dict)
model.eval()
print('Checkpoint loaded.')

N_TEST = 1401
print(f'\nBuilding N={N_TEST} sample...')
sample, _ = build_sample_b1(N_TEST, seed=0, material='neo_hookean', Lx=args.Lx, Ly=args.Ly,
                             solve_fem=False)

print('\nRunning eager baseline + torch.compile + TF32 attempts (this can take ~15-20 min -- '
      'the FIRST compiled call triggers real compilation, not counted in the timing)...')
result = profile_with_torch_compile(sample, model, args, device, dtype,
                                     n_repeats=200, n_warmup=20, compile_warmup=5,
                                     try_tf32=True)

OUT_JSON = f'{R}/no_inference_torch_compile_N1401.json'
with open(OUT_JSON, 'w') as f:
    json.dump(result, f, indent=2)
print('Saved:', OUT_JSON)

print('\n' + '=' * 70)
print('RESULT -- torch.compile attempt at N=1401')
print('=' * 70)
print(json.dumps(result, indent=2))
if result['compile_succeeded']:
    print(f"\ntorch.compile: {result['compiled_ms_per_sample']:.4f} ms/sample vs. eager "
          f"{result['eager_ms_per_sample']:.4f} ms/sample -- {result['speedup_vs_eager']:.2f}x, "
          f"output relative difference vs. eager: {result['compiled_vs_eager_rel_diff']:.3e}")
    print("If the relative difference above is at floating-point noise level (~1e-5 or "
          "tighter for fp32), this speedup is a real, correctness-preserving optimization "
          "and the compiled number, not 2.29s, should be the one considered for finalizing.")
else:
    print(f"\ntorch.compile did not produce a usable result on this model: {result['error']}")
    print("Eager mode's 2.29s stays the answer -- this was a genuine attempt, honestly "
          "reported, not assumed to succeed.")

if result['eager_tf32_ms_per_sample'] is not None:
    print(f"\neager + TF32: {result['eager_tf32_ms_per_sample']:.4f} ms/sample -- "
          f"{result['speedup_tf32_vs_eager']:.2f}x vs. strict-fp32 eager, output relative "
          f"difference: {result['eager_tf32_vs_eager_rel_diff']:.3e}")
    if result['compiled_tf32_ms_per_sample'] is not None:
        print(f"torch.compile + TF32: {result['compiled_tf32_ms_per_sample']:.4f} ms/sample -- "
              f"{result['speedup_compiled_tf32_vs_eager']:.2f}x vs. strict-fp32 eager, output "
              f"relative difference: {result['compiled_tf32_vs_eager_rel_diff']:.3e}")
    print("TF32 trades precision (19-bit mantissa) for speed on Ampere+ tensor cores -- "
          "check the relative differences above are acceptable (comparable to bf16's own "
          "~1e-2 self-consistency gap, or tighter) before treating a TF32 number as the "
          "one to finalize.")
elif result['tf32_error'] is not None:
    print(f"\nTF32 test did not produce a usable result: {result['tf32_error']}")

# ---- Figure ------------------------------------------------------------
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from plot_style import PRIMARY, SECONDARY, add_bar_labels

fig, ax = plt.subplots(figsize=(7, 4.5), dpi=200)
labels = ['eager\n(fp32)']
values = [result['eager_ms_per_sample']]
colors = [PRIMARY]
if result['compile_succeeded']:
    labels.append('torch.compile')
    values.append(result['compiled_ms_per_sample'])
    colors.append(SECONDARY)
if result['eager_tf32_ms_per_sample'] is not None:
    labels.append('eager\n+ TF32')
    values.append(result['eager_tf32_ms_per_sample'])
    colors.append('#27AE60')
if result['compiled_tf32_ms_per_sample'] is not None:
    labels.append('compile\n+ TF32')
    values.append(result['compiled_tf32_ms_per_sample'])
    colors.append('#C0392B')
bars = ax.bar(labels, values, color=colors)
ax.set_ylabel('ms/sample')
title = 'NO inference, N=1401: optimization attempts'
if not result['compile_succeeded']:
    title += '\n(torch.compile FAILED -- see printed error)'
ax.set_title(title)
ax.grid(True, axis='y', alpha=0.25)
add_bar_labels(ax, bars, fmt='{:.1f}')
fig.tight_layout()
FIG_PATH = f'{R}/fig_no_inference_torch_compile_N1401.png'
fig.savefig(FIG_PATH)
print('\nSaved figure:', FIG_PATH)
